# 00. 공통 설정 및 환경 확인
설정 우선순위: YAML < 저장한 공통 설정 < 현재 노트북 첫 셀.

## 1. 사용자 설정
기본 5 seed와 세 head입니다. 데이터/완료 checkpoint를 수정하지 않습니다.

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
# 사용자 설정은 여기에서 지정합니다. YAML은 누락값의 기본값입니다.
SEEDS = [8972, 8973, 8974, 8975, 8976]  # 후속 baseline 학습 5회
HEADS = ["arcface", "adaface", "magface"]
PARENT_SEED = 8972  # 공통 부모는 한 번만 생성
PRETRAIN_EPOCHS = 40  # 사전학습 일정은 본 연구 선택, baseline 40과 별개
PARENT_CHECKPOINT = None  # None: 02의 명시된 공통 부모 경로
PARENT_SHA256 = None  # None: 해당 부모의 _hash.json 사용. 자동 최신 선택 없음
FINETUNE_DATASETS = ["tinyface", "survface"]  # SCFace 미확보: 실행 금지
FINETUNE_PROFILES = default_profiles(BASE_DIR)  # 03의 training split과 출처 manifest
KEEP_RAW_RESULTS = True
NUM_WORKERS = 4  # 실제 후속 batch 구간 실측 권장. 01의 표본 검사는 0 worker
PREFETCH_FACTOR = 4
PERSISTENT_WORKERS = True
TORCH_THREADS = 2
LOSS_SYNC_BATCHES = 20  # loss의 CPU 전송을 묶음 처리; 학습/가중치 갱신은 매 batch 수행
RECOVERY_MINUTES = 10  # step 경계의 최신 재개 캐시 1개 유지
DEVICE = "cuda:0"
DATASET = dict(name="VGGFace2", raw_root="F:/FaceX-Zoo/VGGFace2",
    train_list="F:/FaceX-Zoo/fine-tune/data/train_list.txt",
    val_list="F:/FaceX-Zoo/fine-tune/data/val_list.txt",
    input_size=[112,112], pixel_mean=127.5, pixel_std=128.0, num_classes=8631)
BACKBONE = dict(type="MobileFaceNet", feat_dim=1024, out_h=7, out_w=7, dropout_rate=0.6)
TRAINING = dict(optimizer="adam", lr=0.001, weight_decay=0.0001, epoches=40,
    milestones=[10,20,30], lr_decay_ratio=0.1, batch_size=64, save_freq=1, print_freq=20)
HEAD_PARAMETERS = {
 "arcface":dict(type="ArcFace",feat_dim=1024,num_class=8631,margin_arc=0.6,margin_am=0.0,scale=64.0),
 "adaface":dict(type="AdaFace",feat_dim=1024,num_class=8631,m=0.4,h=0.333,scale=64.0,t_alpha=0.01),
 "magface":dict(type="MagFace",feat_dim=1024,num_class=8631,margin_am=0.0,scale=64.0,l_a=10.0,u_a=110.0,l_margin=0.45,u_margin=0.8,lamda=20.0)}
# 이전 중단 run과 별도 출력 namespace. 기존 파일을 덮어쓰지 않습니다.
PATHS = dict(checkpoint_dir="F:/FaceX-Zoo/fine-tune/checkpoints/shared_parent_v2",
 export_dir="F:/FaceX-Zoo/fine-tune/checkpoints/exported/shared_parent_v2",
 log_dir="F:/FaceX-Zoo/fine-tune/logs", report_dir="F:/FaceX-Zoo/fine-tune/reports/campaigns",
 cache_dir="F:/FaceX-Zoo/fine-tune/data/index_cache")
REUSE_COMPLETED = True
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
 test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
 meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS})


## 2. 설정 저장

In [2]:
print(save_settings(SETTINGS_PATH, settings))

F:\FaceX-Zoo\fine-tune\data\notebook_settings.json


## 3. 실제 환경 확인

In [3]:
from modules.notebook_workflow import inspect_environment
environment = inspect_environment(BASE_DIR, settings)
print(json.dumps(environment, ensure_ascii=False, indent=2))

{
  "python": "3.11.9",
  "torch": "2.7.1+cu118",
  "cuda_available": true,
  "gpu": "NVIDIA GeForce GTX 1080 Ti",
  "branch": "add-adaface",
  "branch_matches": true,
  "train_archive": {
    "exists": true,
    "bytes": 37909212476
  },
  "test_archive": {
    "exists": true,
    "bytes": 2028325313
  },
  "meta_csv": {
    "exists": true,
    "bytes": 335000
  },
  "free_bytes": 1451851735040,
  "training_identities": 8631,
  "identity_count_matches": true,
  "archive_integrity": "not_checked",
  "all_images_decode": "not_checked",
  "readiness": "partial_environment_check_only"
}


## 4. 실행 계획

In [4]:
for stage in ("pretrain","finetune","export"):
    plan=plan_matrix(settings,stage)
    print(stage,len(plan),[r["run_id"] for r in plan])
print("공통 부모 1개. 선택 데이터셋별 세 head × SEEDS. 공유 부모는 본 연구 선택입니다.")

pretrain 1 ['pretrain_arcface_seed8972']
finetune 30 ['finetune_tinyface_arcface_seed8972', 'finetune_tinyface_adaface_seed8972', 'finetune_tinyface_magface_seed8972', 'finetune_tinyface_arcface_seed8973', 'finetune_tinyface_adaface_seed8973', 'finetune_tinyface_magface_seed8973', 'finetune_tinyface_arcface_seed8974', 'finetune_tinyface_adaface_seed8974', 'finetune_tinyface_magface_seed8974', 'finetune_tinyface_arcface_seed8975', 'finetune_tinyface_adaface_seed8975', 'finetune_tinyface_magface_seed8975', 'finetune_tinyface_arcface_seed8976', 'finetune_tinyface_adaface_seed8976', 'finetune_tinyface_magface_seed8976', 'finetune_survface_arcface_seed8972', 'finetune_survface_adaface_seed8972', 'finetune_survface_magface_seed8972', 'finetune_survface_arcface_seed8973', 'finetune_survface_adaface_seed8973', 'finetune_survface_magface_seed8973', 'finetune_survface_arcface_seed8974', 'finetune_survface_adaface_seed8974', 'finetune_survface_magface_seed8974', 'finetune_survface_arcface_seed897

## 5. 확인 범위

In [5]:
print("부분 환경 점검입니다. 압축 전체 무결성·전체 이미지 decode·학습·FIQA 평가는 미실행입니다.")
print("안내:", BASE_DIR / "fine-tune/README.md")

부분 환경 점검입니다. 압축 전체 무결성·전체 이미지 decode·학습·FIQA 평가는 미실행입니다.
안내: F:\FaceX-Zoo\fine-tune\README.md
